# GRPO on GSM8K - Google Colab Training Notebook

This notebook runs the training pipeline for **GRPO on a Small LLM (Qwen2.5-1.5B/0.5B)** on a free-tier T4 GPU with checkpoint saving to Google Drive.

### 1. Mount Google Drive for Persistent Checkpoints

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/grpo-gsm8k'
os.makedirs(f'{DRIVE_DIR}/outputs', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)

### 2. Verify GPU

In [ ]:
!nvidia-smi

### 3. Install Dependencies

In [ ]:
!pip install -q trl peft transformers datasets accelerate pyyaml wandb pytest bitsandbytes

### 4. Run Reward Unit Tests

In [ ]:
!pytest

### 5. Evaluate Untouched Baselines (Dev split)

In [ ]:
!python src/evaluate.py --base Qwen/Qwen2.5-0.5B-Instruct --split dev --out results/base_0.5b
!python src/evaluate.py --base Qwen/Qwen2.5-1.5B-Instruct --split dev --out results/base_1.5b

### 6. Run 50-Step Pilot on 1.5B Model
Measure seconds per step and peak memory before launching full runs.

In [ ]:
!python src/train.py \
    --run-name pilot_1.5b \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --reward format_correct \
    --max-steps 50 \
    --output-root /content/drive/MyDrive/grpo-gsm8k/outputs

### 7. Launch Main Training Run (Checkpoints to Drive)

In [ ]:
!python src/train.py \
    --run-name q15_format_correct_s42 \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --reward format_correct \
    --seed 42 \
    --max-steps 300 \
    --output-root /content/drive/MyDrive/grpo-gsm8k/outputs